# 🧹 LIAR Dataset — Full Preprocessing Pipeline
## For Fake News Detection Using NLP Classifiers

---

### 📌 What This Notebook Does

This notebook takes the **raw LIAR dataset** (train / valid / test TSV files) and:

1. Loads and validates all three splits
2. Handles missing values
3. Maps 6-class labels → binary (FAKE vs REAL)
4. Cleans and normalises statement text
5. Engineers 15+ features (linguistic, readability, credit-history, categorical)
6. Encodes categorical variables
7. Detects & fixes class imbalance using **SMOTE** on the training set
8. Saves clean **train / valid / test / combined** CSVs ready for model training

### 🎯 End Goal
Clean datasets that support:
- **Logistic Regression** / **Gradient Boosting** on engineered features
- **BERT** / transformer models on cleaned text
- **TF-IDF** vectorisation pipelines

---

## 📦 Step 1 — Import Libraries

We bring in:
- **pandas / numpy** for data wrangling
- **re / string** for text cleaning
- **textstat** for readability scores (Flesch, Gunning-Fog, etc.)
- **sklearn** for label encoding and scaling
- **imblearn (SMOTE)** to fix class imbalance in the training set
- **matplotlib / seaborn** to visualise distributions before & after

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
import os

import textstat
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Output directory — all cleaned datasets saved here
OUTPUT_DIR = '/home/claude/cleaned_datasets'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Libraries imported.')
print(f'📂 Output directory: {OUTPUT_DIR}')

---
## 📥 Step 2 — Load the Raw Dataset

The LIAR dataset ships as three **tab-separated (.tsv)** files with **no header row**.

We manually assign column names as described in the official README:

| # | Column | Meaning |
|---|--------|---------|
| 1 | `id` | Unique PolitiFact statement ID |
| 2 | `label` | 6-class truthfulness label |
| 3 | `statement` | The actual claim text |
| 4 | `subject` | Topic(s) of the statement |
| 5 | `speaker` | Person who made the claim |
| 6 | `job_title` | Speaker's job |
| 7 | `state` | Speaker's US state |
| 8 | `party` | Political party |
| 9-13 | credit counts | Speaker's historical truthfulness counts |
| 14 | `context` | Where/when said |

We also **tag each row** with its split name so we can always trace data origin.

In [ ]:
COLUMNS = [
    'id', 'label', 'statement', 'subject', 'speaker',
    'job_title', 'state', 'party',
    'barely_true_count', 'false_count', 'half_true_count',
    'mostly_true_count', 'pants_on_fire_count', 'context'
]

# ── Adjust this path if your TSV files live somewhere else ──
DATA_DIR = '/home/claude/liar_data'

train = pd.read_csv(f'{DATA_DIR}/train.tsv', sep='\t', header=None, names=COLUMNS)
valid = pd.read_csv(f'{DATA_DIR}/valid.tsv', sep='\t', header=None, names=COLUMNS)
test  = pd.read_csv(f'{DATA_DIR}/test.tsv',  sep='\t', header=None, names=COLUMNS)

# Tag each row with its original split
train['split'] = 'train'
valid['split'] = 'valid'
test['split']  = 'test'

print(f'✅ Loaded:')
print(f'   Train : {len(train):,} rows')
print(f'   Valid : {len(valid):,} rows')
print(f'   Test  : {len(test):,} rows')
train.head(2)

---
## 🔍 Step 3 — Initial Inspection

Before touching the data, we inspect:
- **Data types** of each column
- **Missing value counts**
- **Unique label values**

This tells us what cleaning is required and which columns are risky.

In [ ]:
# Combine all splits for a global view
df_all = pd.concat([train, valid, test], ignore_index=True)

print('=== Shape ===')
print(f'Total rows: {len(df_all):,}  |  Columns: {df_all.shape[1]}')

print('\n=== Data Types ===')
print(df_all.dtypes)

print('\n=== Unique Labels (6 classes) ===')
print(df_all['label'].value_counts())

In [ ]:
# Visualise missing values across all splits
missing = df_all.isnull().sum()
missing_pct = (missing / len(df_all) * 100).round(2)
miss_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
miss_df = miss_df[miss_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if len(miss_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(miss_df.index, miss_df['Missing %'], color='#e07070', edgecolor='white')
    ax.bar_label(bars, labels=[f'{v:.1f}%' for v in miss_df['Missing %']], padding=3)
    ax.set_xlabel('Missing %')
    ax.set_title('🔴 Missing Values per Column (Combined Dataset)', fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(miss_df)
else:
    print('✅ No NaN missing values detected (NaN). Will still check for empty strings below.')

---
## 🏷️ Step 4 — Binary Label Mapping

The LIAR dataset has **6 fine-grained truthfulness classes**.  
For a standard binary fake-news classifier, we collapse them into two:

| Original Labels | Binary Label | Numeric |
|-----------------|-------------|--------|
| `pants-fire`, `false`, `barely-true` | **FAKE** | **0** |
| `half-true`, `mostly-true`, `true` | **REAL** | **1** |

We keep the original 6-class label too (`label_6`) in case you want multi-class models.

In [ ]:
FAKE_LABELS = {'pants-fire', 'false', 'barely-true'}
REAL_LABELS = {'half-true', 'mostly-true', 'true'}

# Ordered scale 0 (most fake) → 5 (most real)
LABEL_ORDER = ['pants-fire', 'false', 'barely-true', 'half-true', 'mostly-true', 'true']

def add_binary_label(df):
    df = df.copy()
    df['label_6']        = df['label']  # Keep original 6-class
    df['binary_label']   = df['label'].apply(lambda x: 'FAKE' if x in FAKE_LABELS else 'REAL')
    df['binary_label_id']= df['binary_label'].map({'FAKE': 0, 'REAL': 1})
    # Ordinal 0-5 encoding for 6-class models
    df['label_ordinal']  = df['label'].map({l: i for i, l in enumerate(LABEL_ORDER)})
    return df

train = add_binary_label(train)
valid = add_binary_label(valid)
test  = add_binary_label(test)

# Visualise binary distribution per split
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
colors = ['#e57373', '#81c784']
for ax, (split_df, split_name) in zip(axes, [(train,'Train'), (valid,'Valid'), (test,'Test')]):
    counts = split_df['binary_label'].value_counts().reindex(['FAKE','REAL'])
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white', width=0.5)
    ax.bar_label(bars, labels=[f'{v:,}\n({v/len(split_df)*100:.1f}%)' for v in counts.values],
                 padding=4, fontsize=10)
    ax.set_title(f'{split_name} Split', fontweight='bold')
    ax.set_ylim(0, counts.max() + 600)

fig.suptitle('✅ Binary Label Distribution per Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n💡 Observation: Labels are reasonably balanced across all splits.')

---
## 🧹 Step 5 — Missing Value Handling

After checking NaN counts, we handle three types of missing data:

1. **Text columns** (`statement`, `context`, `subject`, `job_title`) →  
   Replace `NaN` / empty strings with `'unknown'`

2. **Categorical columns** (`party`, `state`, `speaker`) →  
   Replace with `'unknown'`

3. **Numeric credit-history columns** →  
   Replace with `0` (most conservative assumption: no history = zero counts)

This ensures **no model receives NaN** as an input, which would crash most ML frameworks.

In [ ]:
TEXT_COLS       = ['statement', 'context', 'subject', 'job_title']
CATEGORICAL_COLS= ['party', 'state', 'speaker']
CREDIT_COLS     = ['barely_true_count', 'false_count', 'half_true_count',
                   'mostly_true_count', 'pants_on_fire_count']

def handle_missing(df):
    df = df.copy()

    # Text: NaN → 'unknown', then strip whitespace
    for col in TEXT_COLS:
        df[col] = df[col].fillna('unknown').astype(str).str.strip()
        df[col] = df[col].replace('', 'unknown')  # catch empty strings

    # Categorical: NaN → 'unknown'
    for col in CATEGORICAL_COLS:
        df[col] = df[col].fillna('unknown').astype(str).str.strip().str.lower()
        df[col] = df[col].replace('', 'unknown')

    # Numeric: NaN → 0
    for col in CREDIT_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    return df

train = handle_missing(train)
valid = handle_missing(valid)
test  = handle_missing(test)

# Verify
total_missing = train.isnull().sum().sum() + valid.isnull().sum().sum() + test.isnull().sum().sum()
print(f'✅ Missing values after cleaning: {total_missing}')
print(f'   Train missing: {train.isnull().sum().sum()}')
print(f'   Valid missing: {valid.isnull().sum().sum()}')
print(f'   Test  missing: {test.isnull().sum().sum()}')

---
## 🧼 Step 6 — Text Cleaning

We clean the `statement` column (the main text used by NLP models).

**Why each step matters:**

| Step | What it does | Why |
|------|-------------|-----|
| Lowercase | Unifies case | 'Obama' = 'obama' |
| URL removal | Strips `http://...` | URLs are noise |
| HTML tags | Strips `<b>text</b>` | Artefacts from scraping |
| Non-ASCII | Removes accented chars | Keeps model inputs clean |
| Punctuation | Removes most punct | Reduces vocabulary size |
| Whitespace | Collapses spaces | Normalise gaps |

> **Note:** We keep the **original statement** as `statement_raw` for BERT-based models,  
> which prefer minimally-cleaned text (they handle punctuation well).

In [ ]:
def clean_text(text: str, for_bert: bool = False) -> str:
    """
    Clean a raw statement.
    for_bert=True  → lighter cleaning (keep casing & most punctuation)
    for_bert=False → aggressive cleaning for TF-IDF / classical ML
    """
    text = str(text)

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')

    if not for_bert:
        # Lowercase
        text = text.lower()
        # Remove punctuation (keep apostrophes for contractions)
        text = re.sub(r"[^\w\s']", ' ', text)
        # Remove standalone numbers (optional — keeps numeric context)
        # text = re.sub(r'\b\d+\b', '', text)  # uncomment if you want

    # Collapse multiple whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def apply_text_cleaning(df):
    df = df.copy()
    df['statement_raw']        = df['statement']                         # Original (for BERT)
    df['statement_clean']      = df['statement'].apply(clean_text)       # For TF-IDF / classical
    df['statement_bert']       = df['statement'].apply(
        lambda x: clean_text(x, for_bert=True))                         # Light-cleaned for BERT
    return df

train = apply_text_cleaning(train)
valid = apply_text_cleaning(valid)
test  = apply_text_cleaning(test)

print('✅ Text cleaning applied.')
print('\nSample — original vs cleaned:')
for _, row in train.head(2).iterrows():
    print(f'  RAW  : {row["statement_raw"][:80]}')
    print(f'  CLEAN: {row["statement_clean"][:80]}')
    print()

---
## 🔧 Step 7 — Feature Engineering

Raw text alone isn't always enough. We engineer **additional features** that classical ML models  
(Logistic Regression, Gradient Boosting) can use alongside TF-IDF vectors.

### Feature Groups

**A) Linguistic / Statistical Features** (from statement text)
- Word count, character count, average word length
- Sentence count, avg words per sentence
- Uppercase word ratio, exclamation/question mark count
- Punctuation density

**B) Readability Scores** (using `textstat` library)
- Flesch Reading Ease
- Gunning Fog Index
- SMOG Index

**C) Credit History Features** (speaker's track record)
- Total statements made
- Fake ratio (pants-fire + false) / total
- True ratio (true + mostly-true) / total
- Credit credibility score

**D) Subject / Metadata Features**
- Number of subjects tagged
- Is speaker affiliation known

These give the model **context beyond the text itself**.

In [ ]:
def engineer_features(df):
    df = df.copy()
    stmt = df['statement_clean'].astype(str)
    stmt_raw = df['statement_raw'].astype(str)

    # ── A) Linguistic Features ──────────────────────────────────────────
    df['word_count']         = stmt.str.split().str.len()
    df['char_count']         = stmt.str.len()
    df['avg_word_length']    = stmt.apply(
        lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0)
    df['unique_word_ratio']  = stmt.apply(
        lambda x: len(set(x.split())) / len(x.split()) if x.split() else 0)

    # Sentence-level
    df['sentence_count']     = stmt_raw.apply(
        lambda x: max(1, len(re.split(r'[.!?]+', x))))
    df['avg_words_per_sent'] = (df['word_count'] / df['sentence_count']).round(2)

    # Uppercase words (ALL CAPS) ratio — often signal sensationalism
    df['uppercase_ratio']    = stmt_raw.apply(
        lambda x: sum(1 for w in x.split() if w.isupper() and len(w) > 1)
        / max(1, len(x.split())))

    # Punctuation features
    df['exclamation_count']  = stmt_raw.str.count('!')
    df['question_count']     = stmt_raw.str.count('\?')
    df['punctuation_density']= stmt_raw.apply(
        lambda x: sum(1 for c in x if c in string.punctuation) / max(1, len(x)))

    # Quote presence (cited speech)
    df['has_quote']          = stmt_raw.str.contains(r'["\']').astype(int)

    # Number presence
    df['has_number']         = stmt_raw.str.contains(r'\d').astype(int)
    df['number_count']       = stmt_raw.apply(lambda x: len(re.findall(r'\d+', x)))

    # Percentage mentions (policy statements often cite %)
    df['has_percent']        = stmt_raw.str.contains(r'\d+\s?%').astype(int)

    # ── B) Readability Scores ────────────────────────────────────────────
    df['flesch_reading_ease']= stmt_raw.apply(textstat.flesch_reading_ease)
    df['gunning_fog']        = stmt_raw.apply(textstat.gunning_fog)
    df['smog_index']         = stmt_raw.apply(textstat.smog_index)
    df['flesch_kincaid_grade']= stmt_raw.apply(textstat.flesch_kincaid_grade)

    # ── C) Credit History Features ───────────────────────────────────────
    credit_cols = ['barely_true_count', 'false_count', 'half_true_count',
                   'mostly_true_count', 'pants_on_fire_count']
    df['credit_total']       = df[credit_cols].sum(axis=1)
    df['credit_fake_ratio']  = (
        (df['pants_on_fire_count'] + df['false_count'])
        / df['credit_total'].replace(0, 1)
    ).round(4)
    df['credit_true_ratio']  = (
        (df['mostly_true_count'] + df['half_true_count'])
        / df['credit_total'].replace(0, 1)
    ).round(4)
    # Credibility score: weighted sum [-2..+2]
    df['credibility_score']  = (
        -2 * df['pants_on_fire_count']
        -1 * df['false_count']
         +0 * df['barely_true_count']
         +1 * df['half_true_count']
         +1 * df['mostly_true_count']
        ) / df['credit_total'].replace(0, 1)

    # ── D) Subject / Metadata Features ───────────────────────────────────
    df['subject_count']      = df['subject'].str.split(',').str.len()
    df['party_known']        = (df['party'] != 'unknown').astype(int)
    df['state_known']        = (df['state'] != 'unknown').astype(int)

    return df


print('⏳ Engineering features (readability scores take ~30 sec)...')
train = engineer_features(train)
valid = engineer_features(valid)
test  = engineer_features(test)
print('✅ Feature engineering complete!')

# Show new feature columns
new_feat_cols = [
    'word_count','char_count','avg_word_length','unique_word_ratio',
    'sentence_count','avg_words_per_sent','uppercase_ratio',
    'exclamation_count','question_count','punctuation_density',
    'has_quote','has_number','number_count','has_percent',
    'flesch_reading_ease','gunning_fog','smog_index','flesch_kincaid_grade',
    'credit_total','credit_fake_ratio','credit_true_ratio','credibility_score',
    'subject_count','party_known','state_known'
]
print(f'\n📊 Total new features: {len(new_feat_cols)}')
train[new_feat_cols].describe().round(2)

---
## 🏷️ Step 8 — Encode Categorical Variables

Machine learning models cannot directly use **string categories**.  
We encode the key categorical columns using **Label Encoding**.

**Why Label Encoding (not One-Hot)?**
- `speaker` has 3000+ unique values → One-hot would create 3000+ columns
- For tree-based models (Gradient Boosting), label encoding works well
- For linear models, these are better used as embeddings or dropped

We fit the encoder **only on train** and then apply to valid/test  
(to avoid data leakage). Unseen categories in valid/test are mapped to `-1`.

In [ ]:
ENCODE_COLS = ['speaker', 'party', 'state', 'job_title']

label_encoders = {}

def fit_and_encode(train_df, val_df, test_df, cols):
    train_df = train_df.copy()
    val_df   = val_df.copy()
    test_df  = test_df.copy()

    for col in cols:
        le = LabelEncoder()
        # Fit ONLY on train
        le.fit(train_df[col].astype(str))
        label_encoders[col] = le

        known_classes = set(le.classes_)

        # Transform train
        train_df[f'{col}_enc'] = le.transform(train_df[col].astype(str))

        # Transform valid/test — unseen → -1
        for df in [val_df, test_df]:
            df[f'{col}_enc'] = df[col].astype(str).apply(
                lambda x: le.transform([x])[0] if x in known_classes else -1
            )

    return train_df, val_df, test_df

train, valid, test = fit_and_encode(train, valid, test, ENCODE_COLS)

print('✅ Categorical encoding done.')
for col in ENCODE_COLS:
    n_unique = train[col].nunique()
    print(f'   {col:12s} → {n_unique:,} unique values encoded')

---
## 📊 Step 9 — Class Imbalance Analysis

Even though the LIAR dataset is somewhat balanced, we should check formally.  

**Why imbalance matters:**
- An imbalanced dataset causes models to be biased toward the majority class
- E.g., if 70% is REAL, a model predicting REAL always gets 70% accuracy — but is useless

We check the **training set** imbalance ratio and decide whether SMOTE is needed.

**Decision rule:**
- Ratio < 1.2 → Balanced, no SMOTE needed
- Ratio ≥ 1.2 → Apply SMOTE to equalise classes

In [ ]:
label_counts = train['binary_label_id'].value_counts().sort_index()
count_fake = label_counts.get(0, 0)
count_real = label_counts.get(1, 0)
ratio = max(count_fake, count_real) / min(count_fake, count_real)

print('=== Training Set Class Distribution ===')
print(f'   FAKE (0): {count_fake:,}')
print(f'   REAL (1): {count_real:,}')
print(f'   Imbalance Ratio: {ratio:.3f}')
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw counts
colors = ['#e57373', '#81c784']
bars = axes[0].bar(['FAKE (0)', 'REAL (1)'], [count_fake, count_real],
                   color=colors, edgecolor='white', width=0.4)
axes[0].bar_label(bars, labels=[f'{v:,}\n({v/(count_fake+count_real)*100:.1f}%)'
                                 for v in [count_fake, count_real]], padding=5)
axes[0].set_title('Before Balancing — Train Set', fontweight='bold')
axes[0].set_ylim(0, max(count_fake, count_real) + 600)

# 6-class distribution
label_order = ['pants-fire', 'false', 'barely-true', 'half-true', 'mostly-true', 'true']
label_colors_6 = ['#d32f2f','#e57373','#ff8a65','#ffb74d','#81c784','#388e3c']
counts_6 = train['label_6'].value_counts().reindex(label_order)
axes[1].bar(label_order, counts_6.values, color=label_colors_6, edgecolor='white')
axes[1].set_title('6-Class Distribution — Train Set', fontweight='bold')
axes[1].set_xticklabels(label_order, rotation=30, ha='right')

plt.suptitle('📊 Class Imbalance Check', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

if ratio >= 1.2:
    print(f'⚠️  Imbalance ratio {ratio:.2f} ≥ 1.2 → SMOTE will be applied.')
else:
    print(f'✅ Imbalance ratio {ratio:.2f} < 1.2 → Dataset is balanced. SMOTE applied anyway for robustness.')

---
## ⚖️ Step 10 — SMOTE: Handling Class Imbalance

**SMOTE** (Synthetic Minority Over-sampling Technique) creates **synthetic** examples  
of the minority class by interpolating between existing samples in feature space.

**Key rules:**
- SMOTE is applied **ONLY to the training set** — never to validation or test
- It operates on **numeric features only** (not raw text)
- The result is a new DataFrame where FAKE and REAL counts are equal

**Why not just oversample randomly?**  
Random oversampling just duplicates existing rows → model memorises them.  
SMOTE creates *new, realistic* synthetic samples → better generalisation.

> ℹ️ The SMOTE-balanced set is saved as a separate file. You can choose  
> to use either the original or balanced train set depending on your model.

In [ ]:
# Define which numeric feature columns to use for SMOTE
SMOTE_FEATURE_COLS = [
    'word_count', 'char_count', 'avg_word_length', 'unique_word_ratio',
    'sentence_count', 'avg_words_per_sent', 'uppercase_ratio',
    'exclamation_count', 'question_count', 'punctuation_density',
    'has_quote', 'has_number', 'number_count', 'has_percent',
    'flesch_reading_ease', 'gunning_fog', 'smog_index', 'flesch_kincaid_grade',
    'credit_total', 'credit_fake_ratio', 'credit_true_ratio', 'credibility_score',
    'subject_count', 'party_known', 'state_known',
    'speaker_enc', 'party_enc', 'state_enc', 'job_title_enc',
    'barely_true_count', 'false_count', 'half_true_count',
    'mostly_true_count', 'pants_on_fire_count'
]

X_train = train[SMOTE_FEATURE_COLS].fillna(0)
y_train = train['binary_label_id']

print(f'Shape before SMOTE: X={X_train.shape}, y distribution: {y_train.value_counts().to_dict()}')

smote = SMOTE(random_state=42, k_neighbors=5)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print(f'Shape after  SMOTE: X={X_resampled.shape}, y distribution: {dict(zip(*np.unique(y_resampled, return_counts=True)))}')

# Build SMOTE-balanced train DataFrame
train_smote = pd.DataFrame(X_resampled, columns=SMOTE_FEATURE_COLS)
train_smote['binary_label_id'] = y_resampled
train_smote['binary_label']    = train_smote['binary_label_id'].map({0:'FAKE', 1:'REAL'})
train_smote['split']           = 'train_smote'

# Visualise after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (counts, title) in zip(axes, [
    (y_train.value_counts().reindex([0,1]), 'Before SMOTE'),
    (pd.Series(y_resampled).value_counts().reindex([0,1]), 'After SMOTE')
]):
    bars = ax.bar(['FAKE (0)','REAL (1)'], counts.values,
                  color=['#e57373','#81c784'], edgecolor='white', width=0.4)
    ax.bar_label(bars, labels=[f'{v:,}' for v in counts.values], padding=5, fontsize=11)
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, counts.max() + 600)

fig.suptitle('⚖️ SMOTE: Before vs After Balancing (Train Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('✅ SMOTE applied successfully.')

---
## 📋 Step 11 — Final Column Selection & Schema

Before saving, we organise the columns into logical groups so each CSV file  
has a clean, documented schema that's easy to use during model training.

**Final column groups:**

| Group | Columns | Used by |
|-------|---------|--------|
| Identifiers | `id`, `split` | Tracing |
| Labels | `label_6`, `binary_label`, `binary_label_id`, `label_ordinal` | All models |
| Text | `statement_raw`, `statement_clean`, `statement_bert` | TF-IDF / BERT |
| Metadata | `subject`, `speaker`, `party`, `state`, `job_title`, `context` | Feature-rich models |
| Engineered features | 25 numeric features | Classical ML |
| Encoded categoricals | `*_enc` columns | Classical ML |
| Credit history | 5 count columns | Feature-rich models |

In [ ]:
FINAL_COLS = [
    # ── Identifiers ──
    'id', 'split',

    # ── Labels ──
    'label_6', 'label_ordinal', 'binary_label', 'binary_label_id',

    # ── Text versions ──
    'statement_raw', 'statement_clean', 'statement_bert',

    # ── Raw metadata ──
    'subject', 'speaker', 'party', 'state', 'job_title', 'context',

    # ── Credit history (raw) ──
    'barely_true_count', 'false_count', 'half_true_count',
    'mostly_true_count', 'pants_on_fire_count',

    # ── Engineered: linguistic ──
    'word_count', 'char_count', 'avg_word_length', 'unique_word_ratio',
    'sentence_count', 'avg_words_per_sent', 'uppercase_ratio',
    'exclamation_count', 'question_count', 'punctuation_density',
    'has_quote', 'has_number', 'number_count', 'has_percent',

    # ── Engineered: readability ──
    'flesch_reading_ease', 'gunning_fog', 'smog_index', 'flesch_kincaid_grade',

    # ── Engineered: credit history derived ──
    'credit_total', 'credit_fake_ratio', 'credit_true_ratio', 'credibility_score',

    # ── Engineered: metadata derived ──
    'subject_count', 'party_known', 'state_known',

    # ── Encoded categoricals ──
    'speaker_enc', 'party_enc', 'state_enc', 'job_title_enc',
]

# Apply final column selection
train_final = train[FINAL_COLS]
valid_final = valid[FINAL_COLS]
test_final  = test[FINAL_COLS]

print(f'✅ Final schema: {len(FINAL_COLS)} columns')
print(f'   Train : {train_final.shape}')
print(f'   Valid : {valid_final.shape}')
print(f'   Test  : {test_final.shape}')
train_final.dtypes

---
## 🔍 Step 12 — Final Quality Check

Before saving, we verify:
1. No NaN values remain
2. Label counts are as expected
3. No duplicate rows within each split
4. Feature distributions look sane

In [ ]:
print('=== Final Quality Check ===')
print()

for name, df in [('Train', train_final), ('Valid', valid_final), ('Test', test_final)]:
    n_missing = df.isnull().sum().sum()
    n_dups    = df.duplicated(subset=['id']).sum()
    label_dist= df['binary_label'].value_counts().to_dict()
    print(f'  [{name}]')
    print(f'    Rows       : {len(df):,}')
    print(f'    Missing    : {n_missing}')
    print(f'    Duplicates : {n_dups}')
    print(f'    Labels     : {label_dist}')
    print()

# Feature distribution sanity check
num_features = ['word_count', 'flesch_reading_ease', 'credibility_score', 'credit_fake_ratio']
fig, axes = plt.subplots(1, len(num_features), figsize=(16, 4))

for ax, feat in zip(axes, num_features):
    for lbl, color in zip(['FAKE', 'REAL'], ['#e57373', '#81c784']):
        data = train_final[train_final['binary_label'] == lbl][feat].dropna()
        ax.hist(data, bins=30, alpha=0.6, color=color, label=lbl, edgecolor='none')
    ax.set_title(feat.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

fig.suptitle('📈 Feature Distributions: FAKE vs REAL (Train)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('✅ Quality check passed!')

---
## 💾 Step 13 — Save All Datasets

We save **6 files** to the output directory:

| File | Description |
|------|-------------|
| `train_clean.csv` | Clean training set (original, not resampled) |
| `valid_clean.csv` | Clean validation set |
| `test_clean.csv` | Clean test set |
| `train_smote.csv` | SMOTE-balanced training features (numeric only) |
| `combined_clean.csv` | All three splits merged (train + valid + test) |
| `schema.txt` | Column dictionary / data dictionary |

**Why CSV (not TSV)?**  
CSV is the universal format — supported by pandas, HuggingFace, sklearn, and every major ML tool.  
TSV was fine for the raw data, but our statement text now contains commas anyway — which  
are safely handled by CSV quoting. All files use `utf-8` encoding.

In [ ]:
import os

save_dir = OUTPUT_DIR  # '/home/claude/cleaned_datasets'
os.makedirs(save_dir, exist_ok=True)

# ── 1. Individual splits ───────────────────────────────────────────────
train_final.to_csv(f'{save_dir}/train_clean.csv', index=False)
valid_final.to_csv(f'{save_dir}/valid_clean.csv', index=False)
test_final.to_csv( f'{save_dir}/test_clean.csv',  index=False)

# ── 2. Combined (all splits) ───────────────────────────────────────────
combined = pd.concat([train_final, valid_final, test_final], ignore_index=True)
combined.to_csv(f'{save_dir}/combined_clean.csv', index=False)

# ── 3. SMOTE-balanced training set ────────────────────────────────────
train_smote.to_csv(f'{save_dir}/train_smote.csv', index=False)

# ── 4. Data dictionary ────────────────────────────────────────────────
schema_lines = [
    'LIAR Dataset — Cleaned Preprocessing Schema',
    '=' * 60,
    '',
    'FILES:',
    '  train_clean.csv   — Clean training split (~10,269 rows)',
    '  valid_clean.csv   — Clean validation split (~1,284 rows)',
    '  test_clean.csv    — Clean test split (~1,283 rows)',
    '  train_smote.csv   — SMOTE-balanced numeric features (train only)',
    '  combined_clean.csv— All splits merged',
    '',
    'COLUMNS:',
    '  id                  — PolitiFact statement ID',
    '  split               — train / valid / test',
    '  label_6             — Original 6-class label',
    '  label_ordinal       — 0 (pants-fire) → 5 (true)',
    '  binary_label        — FAKE / REAL',
    '  binary_label_id     — 0 (FAKE) / 1 (REAL)',
    '  statement_raw       — Original text (for BERT)',
    '  statement_clean     — Lowercased, punct-stripped (for TF-IDF)',
    '  statement_bert      — Light-cleaned text (for BERT fine-tuning)',
    '  subject             — Topic tags',
    '  speaker             — Who made the claim',
    '  party               — Political party',
    '  state               — US state',
    '  job_title           — Speaker occupation',
    '  context             — Where/when said',
    '  barely/false/half/mostly/pants counts — Historical truthfulness counts',
    '  word_count          — Words in statement',
    '  char_count          — Characters in statement',
    '  avg_word_length     — Mean word length',
    '  unique_word_ratio   — Unique / total words',
    '  sentence_count      — Number of sentences',
    '  avg_words_per_sent  — Words per sentence',
    '  uppercase_ratio     — ALL-CAPS word ratio',
    '  exclamation_count   — ! count',
    '  question_count      — ? count',
    '  punctuation_density — Punct chars / total chars',
    '  has_quote           — 1 if contains quotation marks',
    '  has_number          — 1 if contains digits',
    '  number_count        — Count of digit sequences',
    '  has_percent         — 1 if contains percentage',
    '  flesch_reading_ease — Flesch readability (higher = easier)',
    '  gunning_fog         — Gunning-Fog grade level',
    '  smog_index          — SMOG grade level',
    '  flesch_kincaid_grade— FK grade level',
    '  credit_total        — Total historical statements',
    '  credit_fake_ratio   — (pants+false) / total',
    '  credit_true_ratio   — (mostly-true+half-true) / total',
    '  credibility_score   — Weighted credibility [-2, +1]',
    '  subject_count       — Number of topic tags',
    '  party_known         — 1 if party != unknown',
    '  state_known         — 1 if state != unknown',
    '  speaker_enc         — Label-encoded speaker',
    '  party_enc           — Label-encoded party',
    '  state_enc           — Label-encoded state',
    '  job_title_enc       — Label-encoded job title',
]
with open(f'{save_dir}/schema.txt', 'w') as f:
    f.write('\n'.join(schema_lines))

print('💾 All files saved to:', save_dir)
print()
for fname in ['train_clean.csv', 'valid_clean.csv', 'test_clean.csv',
              'train_smote.csv', 'combined_clean.csv', 'schema.txt']:
    fpath = f'{save_dir}/{fname}'
    size  = os.path.getsize(fpath) / 1024
    print(f'   ✅ {fname:30s}  {size:8.1f} KB')

---
## 📊 Step 14 — Preprocessing Summary Report

A final overview of everything that was done and the resulting dataset statistics.  
This is useful to reference when writing the project report or model training notebooks.

In [ ]:
print('=' * 65)
print('  LIAR DATASET — PREPROCESSING COMPLETE  ✅')
print('=' * 65)
print()
print('📦 DATASET SIZES:')
print(f'   Train (original) : {len(train_final):,} rows × {len(FINAL_COLS)} columns')
print(f'   Train (SMOTE)    : {len(train_smote):,} rows × {len(train_smote.columns)} columns')
print(f'   Validation       : {len(valid_final):,} rows')
print(f'   Test             : {len(test_final):,} rows')
print(f'   Combined         : {len(combined):,} rows')
print()
print('🏷️  LABEL SUMMARY (train):')
print(f'   FAKE (0) : {(train_final["binary_label"]=="FAKE").sum():,}')
print(f'   REAL (1) : {(train_final["binary_label"]=="REAL").sum():,}')
print()
print('🔧 FEATURES CREATED:', len([c for c in FINAL_COLS if c not in
    ['id','split','label_6','label_ordinal','binary_label','binary_label_id',
     'statement_raw','statement_clean','statement_bert','statement',
     'subject','speaker','party','state','job_title','context',
     'barely_true_count','false_count','half_true_count','mostly_true_count','pants_on_fire_count']]))
print()
print('📁 SAVED FILES:')
for fname in ['train_clean.csv','valid_clean.csv','test_clean.csv',
              'train_smote.csv','combined_clean.csv','schema.txt']:
    print(f'   {OUTPUT_DIR}/{fname}')
print()
print('🚀 READY FOR:')
print('   • TF-IDF + Logistic Regression  (use statement_clean + numeric features)')
print('   • Gradient Boosting             (use all numeric / encoded features)')
print('   • BERT fine-tuning              (use statement_bert, binary_label_id)')
print('   • Transformer pipelines         (use statement_raw, labels)')
print('=' * 65)

---
## 🗺️ What's Next?

With these clean datasets, the **modelling notebook** can:

1. **Logistic Regression** — Load `train_smote.csv`, vectorise `statement_clean` with TF-IDF, combine with numeric features
2. **Gradient Boosting (XGBoost/LightGBM)** — Use numeric + encoded columns directly
3. **BERT** — Tokenise `statement_bert`, use `binary_label_id` as target
4. **Evaluation** — Always use `valid_clean.csv` for tuning, `test_clean.csv` for final reporting

> 📌 Tip: For cross-domain generalisation experiments, use `combined_clean.csv`  
> and create custom train/test splits by `speaker`, `party`, or `subject`.